# Step 6: Source Detection & Deblending

Before measuring shapes, we must find galaxies in the coadd and separate
overlapping sources. At LSST depth, **~58% of objects are blended**
(Bosch et al. 2018) — deblending is essential.

**LSST tasks:**
- `lsst.meas.algorithms.detection.SourceDetectionTask`
- `lsst.meas.deblender.SourceDeblendTask` (single-band)
- `lsst.meas.extensions.scarlet.ScarletDeblendTask` (multiband)

**Input:** `deepCoadd`  
**Output:** `deepCoadd_deblendedFlux` (deblended images per source)

**Reference:** Bosch et al. (2018) §4.4–4.5

## 6.1 Source Detection

### The algorithm
1. **Smooth** the image with a kernel matched to the PSF FWHM
   (maximizes SNR for point-like sources — matched filter detection)
2. Identify pixels above a **threshold** (default: 5σ above background)
3. Group connected above-threshold pixels into **footprints**
4. Find **peaks** within each footprint — each peak is a candidate source

A single footprint can contain multiple peaks (= a blend).

```python
# In the pipeline:
from lsst.meas.algorithms import SourceDetectionTask

config = SourceDetectionTask.ConfigClass()
config.thresholdValue = 5.0          # 5-sigma detection
config.thresholdType = 'stdev'
config.minPixels = 1                 # minimum footprint size
config.isotropicGrow = True          # grow footprints isotropically

detect_task = SourceDetectionTask(config=config, schema=schema)
result = detect_task.run(table, coadd_exposure)
# result.sources contains the detected footprints and peaks
```

### Footprints
A `Footprint` is the set of pixels belonging to a detected source (or blend).
It stores:
- The pixel **spans** (list of row-start-stop ranges)
- The **peaks** (local maxima within the footprint)
- The **heavy footprint** (actual pixel values, after deblending)

## 6.2 Deblending

### The problem
When two galaxies overlap, their footprints merge. We must assign flux
from each pixel to the correct source — otherwise shapes are biased by
the contaminating neighbor.

### SDSS-like deblender (single-band)
`lsst.meas.deblender.SourceDeblendTask`

1. For each peak in a multi-peak footprint, create a **template**
   by tracing the symmetric profile around that peak
2. Assign flux to each child source proportionally to the templates:
   $$f_i(\vec{x}) = \frac{T_i(\vec{x})}{\sum_j T_j(\vec{x})} \cdot I(\vec{x})$$
3. Each child gets a **HeavyFootprint** with its assigned pixel values

### Scarlet (multiband, preferred)
`lsst.meas.extensions.scarlet.ScarletDeblendTask`

A much more sophisticated approach using **constrained matrix factorization**:

$$I_{\text{band}}(\vec{x}) = \sum_k A_k(\text{band}) \cdot S_k(\vec{x}) * \text{PSF}_{\text{band}}(\vec{x})$$

where for each source $k$:
- $A_k$ = spectral energy distribution (SED) — how bright in each band
- $S_k$ = morphology — non-parametric spatial profile

Constraints:
- **Non-negative** flux (astrophysical)
- **Monotonic** radial profiles (galaxies don't have flux holes)
- **Compact** support (sources are localized)

Scarlet jointly fits all bands, using color information to separate sources
that are spatially overlapping but have different SEDs.

### Why deblending is critical for weak lensing
- Undeblended neighbors bias shape measurements toward the blend direction
- In LSST, >50% of objects are blended — can't just reject them
- The `NoiseReplacer` (in `meas.base`) replaces deblended neighbors with
  noise during measurement, so each galaxy is measured in isolation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, label, find_objects
from scipy.ndimage import maximum_filter

rng = np.random.default_rng(42)

In [ ]:
# --- Simulate detection ---

ny, nx = 200, 200
yy, xx = np.mgrid[:ny, :nx]
psf_fwhm = 3.5  # pixels
psf_sigma = psf_fwhm / 2.355
noise_sigma = 10.0

# Create sources: some isolated, some blended
true_sources = []
scene = np.zeros((ny, nx))

source_params = [
    # (cx, cy, flux, gal_sigma, e1)
    (50, 50, 5000, 4.0, 0.2),      # isolated
    (120, 40, 8000, 5.0, -0.15),    # isolated
    (80, 120, 12000, 6.0, 0.1),     # blend pair: bright
    (88, 126, 4000, 3.0, -0.2),     # blend pair: faint companion
    (160, 160, 3000, 3.5, 0.0),     # isolated, faint
    (40, 160, 15000, 7.0, -0.1),    # isolated, large
    (150, 80, 6000, 4.5, 0.25),     # blend triplet
    (157, 85, 3000, 3.0, -0.1),     # blend triplet
    (145, 88, 2000, 2.5, 0.05),     # blend triplet
]

for (cx, cy, flux, sig, e1) in source_params:
    Qxx = sig**2 * (1 + e1)
    Qyy = sig**2 * (1 - e1)
    gal = flux * np.exp(-0.5*((xx-cx)**2/Qxx + (yy-cy)**2/Qyy))
    scene += gal
    true_sources.append((cx, cy))

# Convolve with PSF
convolved = gaussian_filter(scene, sigma=psf_sigma)
# Add noise
observed = convolved + rng.normal(scale=noise_sigma, size=(ny, nx))

In [ ]:
# Detection: convolve with matched filter, threshold at 5-sigma
matched_filtered = gaussian_filter(observed, sigma=psf_sigma)
# The noise in the filtered image
filtered_noise = noise_sigma / np.sqrt(4 * np.pi * psf_sigma**2)
snr_map = matched_filtered / filtered_noise

# Threshold
threshold = 5.0
detected_mask = snr_map > threshold

# Label connected regions (footprints)
footprint_labels, n_footprints = label(detected_mask)

# Find peaks in each footprint
peak_mask = (maximum_filter(matched_filtered, size=5) == matched_filtered) & detected_mask
peak_y, peak_x = np.where(peak_mask)

print(f"Detected {n_footprints} footprints")
print(f"Found {len(peak_x)} peaks (some footprints have multiple = blends)")
print(f"True sources: {len(true_sources)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Observed image
vmin, vmax = np.percentile(observed, [1, 99.5])
axes[0].imshow(observed, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Observed coadd')

# SNR map with threshold
axes[1].imshow(snr_map, cmap='RdYlBu_r', origin='lower', vmin=0, vmax=20)
axes[1].contour(detected_mask, levels=[0.5], colors='lime', linewidths=1)
axes[1].set_title(f'SNR map (green = {threshold}σ threshold)')

# Footprints + peaks
axes[2].imshow(observed, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[2].contour(footprint_labels, levels=range(1, n_footprints+1),
                colors='cyan', linewidths=0.8, alpha=0.7)
axes[2].scatter(peak_x, peak_y, c='red', s=30, marker='+', lw=1.5,
                label=f'{len(peak_x)} peaks', zorder=5)
for (cx, cy) in true_sources:
    axes[2].scatter(cx, cy, facecolors='none', edgecolors='yellow',
                    s=80, lw=1.5, zorder=4)
axes[2].set_title('Footprints (cyan) + peaks (red) + true (yellow)')
axes[2].legend(loc='upper right')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Source Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/detection_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Simulate simple deblending (template-based) ---

# Focus on the blend pair at (80,120) and (88,126)
blend_cx = [80, 88]
blend_cy = [120, 126]
cutout = observed[100:150, 60:110]
local_yy, local_xx = np.mgrid[:50, :50]

# Create symmetric templates around each peak
templates = []
for cx, cy in zip(blend_cx, blend_cy):
    lx, ly = cx - 60, cy - 100  # local coords
    r2 = (local_xx - lx)**2 + (local_yy - ly)**2
    # Circular template (azimuthal average around peak)
    template = np.exp(-r2 / (2 * 4.0**2))  # approximate
    templates.append(template)

# Normalize templates and assign flux
template_sum = templates[0] + templates[1]
template_sum[template_sum == 0] = 1  # avoid division by zero

child1 = cutout * templates[0] / template_sum
child2 = cutout * templates[1] / template_sum

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
vmin, vmax = np.percentile(cutout, [1, 99])

axes[0].imshow(cutout, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Blended footprint')
axes[0].scatter([20, 28], [20, 26], c='red', marker='+', s=100, lw=2)

axes[1].imshow(templates[0] / template_sum, cmap='RdBu_r', origin='lower',
               vmin=0, vmax=1)
axes[1].set_title('Template weight (source 1)')

axes[2].imshow(child1, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
axes[2].set_title('Deblended child 1')

axes[3].imshow(child2, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
axes[3].set_title('Deblended child 2')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Deblending: Separating Overlapping Sources', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/deblending_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("After deblending, each 'child' source gets a HeavyFootprint")
print("with only its share of the flux. The NoiseReplacer then fills")
print("the neighbor's region with noise during measurement.")

## Detection & Deblending Flags

Important flags in the output catalog:

| Flag | Meaning |
|------|---------|
| `detect_isPrimary` | True for deblended children (not parents) in the primary tract/patch |
| `deblend_nChild` | Number of deblended children (0 = isolated) |
| `deblend_skipped` | Deblending was skipped (too many peaks, too large, etc.) |
| `parent` | ID of the parent footprint (0 = no parent = isolated) |

For science, always filter on `detect_isPrimary == True` to get a clean,
non-duplicated catalog of individual sources.

**Next:** [07_measurement.ipynb](07_measurement.ipynb) — Shape and flux measurement